# 04 - Support Vector Machine
Scaling is inside the leakage-safe pipeline. RBF and linear candidates are selected on validation F1, then test is evaluated once.

In [ ]:
from pathlib import Path
import sys, json
import matplotlib.pyplot as plt
from sklearn.model_selection import ParameterGrid
from sklearn.svm import SVC
from sklearn.metrics import f1_score, recall_score, ConfusionMatrixDisplay
ROOT=Path.cwd()
if not (ROOT/'eeg_app').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from eeg_app.src.experiments import load_prepared, feature_names, classical_pipeline, save_model_result, experiment_dirs, seed_everything
seed_everything(42); frame=load_prepared(ROOT); features=feature_names(frame); train=frame[frame.split.eq('train')]; val=frame[frame.split.eq('validation')]; test=frame[frame.split.eq('test')]
best=None
for params in ParameterGrid({'model__kernel':['linear','rbf'],'model__C':[0.1,1,10],'model__gamma':['scale','auto'],'model__class_weight':[None,'balanced']}):
    model=classical_pipeline(SVC(probability=True,random_state=42),scale=True).set_params(**params); model.fit(train[features],train.label); pred=model.predict(val[features]); key=(f1_score(val.label,pred,zero_division=0),recall_score(val.label,pred,zero_division=0))
    if best is None or key>best[0]: best=(key,params,model)
print('best validation:',best[0],best[1]); model=best[2]; pred=model.predict(test[features]); scores=model.predict_proba(test[features])[:,1]
metrics=save_model_result('svm',model,test,pred,scores,ROOT)
Path(experiment_dirs(ROOT)['results']/'svm_validation.json').write_text(json.dumps({'F1':best[0][0],'Recall':best[0][1],'params':best[1]},indent=2),encoding='utf-8'); print(json.dumps(metrics,indent=2))
ConfusionMatrixDisplay.from_predictions(test.label,pred,display_labels=['normal','seizure']); plt.title('SVM test confusion matrix'); plt.show()